# Frozen contrastive audio replacement, fixed experiment 28

Frozen pinned LAION CLAP, six prompts fixed before extraction, no audio labels or
prompt tuning. Timestamped original audio, half resets and identical audio centers.
Five whole-window holdouts with 60-second purge; PCA and normalization fit only on
remaining training frames. Three seeds per fold and final fit, 60 epochs each.
Selection uses training OOF spellF1; only its winner is scored on reused validation.
Source/weights and prediction hashes remain private; no competition submission.


In [ ]:
from pathlib import Path
import hashlib,json,os,subprocess,sys,torch
assert torch.cuda.is_available()
print('GPU:',torch.cuda.get_device_name(0),flush=True)
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working/possession_clap');WORK.mkdir(parents=True,exist_ok=True)
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
configs=list(INPUT.rglob('clap_input.json'));assert len(configs)==1
FILES=configs[0].parent;plan=json.loads(configs[0].read_text())
roots=[p.parent for p in INPUT.rglob('F1_TARGET.md') if (p.parent/'temporal_possession/clap_study.py').is_file()];assert len(roots)==1
CODE=roots[0]
for relative,digest in plan['source_sha256'].items():assert sha(CODE/relative)==digest
for relative,digest in plan['input_sha256'].items():assert sha(FILES/relative)==digest
FEATURES=FILES/'features.npz';GRAPH=FILES/'graph.npz';REFERENCE=FILES/'reference_audio.npz';OWNER=FILES/'owner_scores.npz'
VIDEOS=list(INPUT.rglob('BAR-ATH_tactical.mp4'));assert len(VIDEOS)==1
VIDEO=VIDEOS[0];assert sha(VIDEO)==plan['video_sha256']
labels=[p.parent for p in INPUT.rglob('train.csv') if p.parent.name=='clap_labels'];assert len(labels)==1
LABELS=labels[0]
for name in ('train','validation'):assert sha(LABELS/f'{name}.csv')==plan['reference_sha256'][name]
original=[p.parent for p in INPUT.rglob('experiment.json') if json.loads(p.read_text()).get('kind')=='team_symmetric_graph_memory_span_ablation'];assert len(original)==1
SOURCE=original[0]
matched=[p.parent for p in INPUT.rglob('experiment.json') if json.loads(p.read_text()).get('kind')=='matched_three_seed_oof_calibration'];assert len(matched)==1
MATCHED=matched[0]
DIRECT=next(p.parent for p in INPUT.rglob('decision.json') if p.parent.name=='clap_direct')
TURNOVERS=next(p.parent for p in INPUT.rglob('decision.json') if p.parent.name=='clap_turnovers')
DEPS=Path('/tmp/barca_clap_python')
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--target',str(DEPS),'-r',str(CODE/'requirements-temporal.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--target',str(DEPS),'--no-deps','-r',str(CODE/'requirements-foundation.txt')],check=True)
env=dict(os.environ,PYTHONPATH=str(DEPS)+os.pathsep+str(CODE),PYTHONDONTWRITEBYTECODE='1',HF_HUB_OFFLINE='1',HF_HUB_DISABLE_XET='1',CUBLAS_WORKSPACE_CONFIG=':4096:8')
def run(*args,network=False):
 current=dict(env)
 if network:current['HF_HUB_OFFLINE']='0'
 subprocess.run([sys.executable,*map(str,args)],cwd=CODE,env=current,check=True)
run('-m','unittest','discover','-s','tests','-v')
print('Full suite passed; starting fixed CLAP extraction.',flush=True)


In [ ]:
AUDIO=WORK/'clap_audio.npz'
run('-m','temporal_possession.clap','--video',VIDEO,'--reference-audio',REFERENCE,'--out',AUDIO,'--device','cuda','--batch-size','16',network=True)


In [ ]:
RUN=WORK/'study'
run('-m','temporal_possession.clap_study','--features',FEATURES,'--audio',AUDIO,'--graph',GRAPH,'--owner-scores',OWNER,
 '--source-study',SOURCE,'--matched-study',MATCHED,'--direct-study',DIRECT,'--turnover-study',TURNOVERS,
 '--train-labels',LABELS/'train.csv','--validation-labels',LABELS/'validation.csv','--out',RUN,'--device','cuda','--bootstrap','1000')


In [ ]:
report=json.loads((RUN/'report.json').read_text());assert not report['experiment']['validation_used_for_selection']
protocol=dict(git_checkpoint=plan['git_checkpoint'],source_sha256=plan['source_sha256'],input_sha256=plan['input_sha256'],
 reference_sha256=plan['reference_sha256'],torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),
 audio_sha256=sha(AUDIO),selected_csv_sha256=sha(RUN/'selected_predictions.csv'),target_audio_labels_added=False,
 reference_files_unchanged=all(sha(LABELS/f'{n}.csv')==plan['reference_sha256'][n] for n in ('train','validation')))
assert protocol['reference_files_unchanged']
(WORK/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')
print('CLAP GPU RUN COMPLETE',flush=True)
